# SVM POLY: evaluasi full-training

Akram — `svm_poly`. Parameter hasil screening sebelumnya: `{'C': 1.0, 'gamma': 'scale', 'degree': 3, 'coef0': 1.0}`.
Setiap model memakai seluruh train fold, tanpa subsampling: stratified 5-fold,
seed 42, feature engineering, tol 0.01, max_iter 1000. Kalibrasi hanya memakai
data train fold. Tidak ada pengiriman Kaggle atau push Git otomatis.

Run berjalan lama di CPU. Log dan checkpoint OOF per fold berada di
`output/experiments/svm_poly/`; hasil lengkap tersimpan setelah notebook selesai.

## Alasan pemilihan hyperparameter

- Kernel `poly` menambahkan interaksi non-linear antarfitur melalui
  `K(x, z) = (gamma * x.T @ z + coef0) ** degree`.
- `degree=3` dipakai untuk menangkap interaksi hingga derajat tiga tanpa menaikkan
  orde polinomial lebih jauh. Derajat yang lebih tinggi dapat memperbesar nilai
  kernel dan kompleksitas model; tidak diasumsikan otomatis menghasilkan skor lebih baik.
- `coef0=1` membuat kernel tidak homogen: ekspansi pangkat tiga memuat komponen
  konstan serta interaksi derajat satu, dua, dan tiga. Jika offset nol, kernel
  hanya memuat komponen derajat tiga. Pilihan ini mempertahankan kontribusi orde rendah.
- `gamma='scale'` memakai `1 / (n_fitur * varians_X)` dari train hasil preprocessing
  di masing-masing fold untuk mengatur skala dot product sebelum dipangkatkan.
  `C=1` menjadi titik kerja penalti pelanggaran margin; pengaruhnya tetap
  bergantung pada skala dan bentuk kernel polynomial.

- `tol=0.01` memberi toleransi pelanggaran kondisi KKT sebesar 0,01 agar optimasi
  tidak mengejar ketelitian yang terlalu mahal untuk eksperimen awal.
  `max_iter=1000` membatasi jumlah putaran jika toleransi belum tercapai;
  tiap putaran memuat maksimal `ceil(n_train / 64)` pembaruan blok QP.
  nilai ini bukan target jumlah iterasi. Kelima model utama pada konfigurasi
  yang dilaporkan mencapai toleransi tersebut.
- `seed=42` membuat pembagian data dan holdout kalibrasi dapat direproduksi;
  seed tidak dipilih dengan membandingkan skor.

- Threshold `0.40` dipilih dari Macro F1 tertinggi pada OOF penuh dalam rentang
  0,20 sampai 0,80 berlangkah 0,01. Nilai ini menyesuaikan keputusan kelas terhadap
  probabilitas terkalibrasi dan metrik Macro F1, tanpa memakai label test.

Konfigurasi yang dibawa dari screening ini menghasilkan OOF penuh **0,7943**,
dibanding **0,7907** pada konfigurasi RBF yang diuji. Notebook ini tidak mengevaluasi
seluruh kombinasi C, gamma, degree, dan coef0 pada data penuh; hasil tersebut
belum membuktikan bahwa konfigurasi ini adalah optimum global.


Optimasi dual memakai `scipy.optimize.minimize(method="SLSQP")` pada blok
maksimal 64 alpha dengan batas `0 <= alpha <= C` dan kendala `sum(alpha * y) = 0`.
Blok kecil menghindari penyimpanan matriks kernel seluruh data. Kalibrasi sigmoid
memakai solver BFGS dari SciPy; kernel, support vector, dan prediksi dihitung sendiri.
`DEVICE='auto'` memakai CUDA untuk perhitungan blok kernel jika tersedia,
lalu CPU sebagai fallback. Gunakan `'cuda'` untuk mewajibkan kernel GPU
atau `'cpu'` untuk seluruh perhitungan di CPU. Solver SciPy berjalan di CPU.
Output dan skor yang tersimpan berasal dari solver SMO CPU sebelumnya;
evaluasi dengan solver SciPy perlu menjalankan ulang notebook.

Referensi solver: [SciPy SLSQP](https://docs.scipy.org/doc/scipy/reference/optimize.minimize-slsqp.html)
dan [SciPy BFGS](https://docs.scipy.org/doc/scipy/reference/optimize.minimize-bfgs.html).


In [1]:
import hashlib
import sys
import time
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from IPython.display import display
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedKFold
from sklearn.svm import SVC
from threadpoolctl import threadpool_limits

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / 'src' / 'models').is_dir())
sys.path.insert(0, str(ROOT / 'src'))
from models import SVM
from preprocessing import SVMPreprocessor, TARGET, clean_train
from utils import (best_threshold, confusion_matrix, cross_val_proba,
                   f1_per_class, fold_scores, load_model, macro_f1, save_model, stratified_kfold)

SEED = 42
DEVICE = 'auto'
N_FOLDS = 5
SCREEN_SIZE = 3000
KERNEL = 'poly'
EXP_NAME = 'svm_poly'
BASE_PARAMS = dict(kernel=KERNEL, tol=0.01, max_iter=1000, seed=SEED, device=DEVICE)
FIG_PATH = ROOT / 'output' / 'figures' / f'{EXP_NAME}.png'
MODEL_PATH = ROOT / 'output' / 'models' / f'{EXP_NAME}.pkl'
SUB_PATH = ROOT / 'output' / 'submissions' / f'submission_{EXP_NAME}.csv'
plt.rcParams.update({'font.size': 11, 'figure.dpi': 110,
                     'axes.spines.top': False, 'axes.spines.right': False})
print('Python', sys.version.split()[0], '| NumPy', np.__version__,
      '| pandas', pd.__version__, '| scikit-learn', sklearn.__version__)

RUN_DIR = ROOT / 'output' / 'experiments' / EXP_NAME
RUN_DIR.mkdir(parents=True, exist_ok=True)


Python 3.12.6 | NumPy 2.2.6 | pandas 2.3.0 | scikit-learn 1.9.0


In [2]:
train_raw = pd.read_csv(ROOT / 'data' / 'train.csv', dtype={'id': str})
train = clean_train(train_raw)
y = train[TARGET].to_numpy()
folds = stratified_kfold(y, n_splits=N_FOLDS, seed=SEED)
assert set(np.unique(y)) == {0, 1}
assert np.array_equal(np.sort(np.concatenate([va for _, va in folds])), np.arange(len(y)))
assert all(not np.intersect1d(tr, va).size for tr, va in folds)
print('Full training:', len(train), '| fold train sizes:', [len(tr) for tr, _ in folds], flush=True)


def report(name, labels, proba, splits, seconds):
    assert proba.shape == labels.shape and np.isfinite(proba).all()
    assert ((0 <= proba) & (proba <= 1)).all()
    threshold, score = best_threshold(labels, proba)
    scores = fold_scores(labels, proba, splits, threshold)
    assert np.isclose(score, f1_score(labels, proba >= threshold, average='macro'))
    return dict(model=name, threshold=threshold, oof_macro_f1=score,
                fold_mean=float(scores.mean()), fold_std=float(scores.std()),
                cv_seconds=seconds)

best_params = {'C': 1.0, 'gamma': 'scale', 'degree': 3, 'coef0': 1.0}


Full training: 27309 | fold train sizes: [21847, 21847, 21847, 21847, 21848]


## From scratch: 5-fold full-training

Checkpoint OOF ditulis secara atomik setelah setiap fold selesai. Nilai NaN menandai fold yang belum selesai.

In [ ]:
scratch_models = []
oof_scratch = np.full(len(y), np.nan)
scratch_times = []
cv_warnings = []

def make_scratch():
    model = SVM(**BASE_PARAMS, **best_params)
    scratch_models.append(model)
    return model

with threadpool_limits(limits=1):
    for k, (tr, va) in enumerate(folds, 1):
        print(f'Mulai full fold {k}/{N_FOLDS}, train={len(tr)}, validasi={len(va)}', flush=True)
        started = time.perf_counter()
        with warnings.catch_warnings(record=True) as messages:
            warnings.simplefilter('always', RuntimeWarning)
            part = cross_val_proba(make_scratch, SVMPreprocessor, train, y, [(tr, va)])
        cv_warnings.extend(messages)
        oof_scratch[va] = part[va]
        assert np.isfinite(oof_scratch[va]).all()
        scratch_times.append(time.perf_counter() - started)
        temp_path = RUN_DIR / 'scratch_progress.tmp.npz'
        np.savez_compressed(temp_path, oof=oof_scratch, y=y, completed_folds=k,
                            fold_seconds=scratch_times,
                            iterations=[m.n_iter_ for m in scratch_models],
                            converged=[m.converged_ for m in scratch_models])
        temp_path.replace(RUN_DIR / 'scratch_progress.npz')
        print(f'Selesai fold {k}: {scratch_times[-1]:.1f}s, '
              f'iterasi={scratch_models[-1].n_iter_}, konvergen={scratch_models[-1].converged_}, '
              f'peringatan={len(messages)}', flush=True)

res_scratch = report('SVM from scratch', y, oof_scratch, folds, sum(scratch_times))
display(pd.DataFrame([res_scratch]).set_index('model').round(4))
display(pd.DataFrame([dict(fold=k, iterations=m.n_iter_, converged=m.converged_, device=m.device_)
                      for k, m in enumerate(scratch_models, 1)]))
print('Peringatan termasuk kalibrasi:', len(cv_warnings))
for message in sorted({str(w.message) for w in cv_warnings}):
    print(message)


## Pembanding scikit-learn

Fold, preprocessing, dan parameter sama; kalibrasi sklearn memakai 5-fold internal dengan ensemble=False.

In [ ]:
class SklearnWrapper:
    def __init__(self, model):
        self.model = model

    def fit(self, X, y):
        self.model.fit(X, y)
        return self

    def predict_proba(self, X):
        return self.model.predict_proba(X)[:, 1]

reference_models = []

def make_reference():
    print(f'Mulai fold sklearn {len(reference_models) + 1}/{N_FOLDS}', flush=True)
    model = SklearnWrapper(CalibratedClassifierCV(
        SVC(kernel=KERNEL, **best_params, tol=BASE_PARAMS['tol'], random_state=SEED),
        method='sigmoid', ensemble=False,
        cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)))
    reference_models.append(model)
    return model

started = time.perf_counter()
with threadpool_limits(limits=1):
    oof_sklearn = cross_val_proba(make_reference, SVMPreprocessor, train, y, folds)
res_sklearn = report('sklearn SVC', y, oof_sklearn, folds,
                     time.perf_counter() - started)
comparison = pd.DataFrame([res_scratch, res_sklearn]).set_index('model')
display(comparison.round(4))
per_fold = pd.DataFrame({
    'fold': np.arange(1, N_FOLDS + 1),
    'from_scratch': fold_scores(y, oof_scratch, folds, res_scratch['threshold']),
    'sklearn': fold_scores(y, oof_sklearn, folds, res_sklearn['threshold']),
})
display(per_fold.round(4))
pred_oof = (oof_scratch >= res_scratch['threshold']).astype(int)
print('Confusion matrix [[TN, FP], [FN, TP]]:')
print(confusion_matrix(y, pred_oof))
print('F1 kelas 0 dan 1:', f1_per_class(y, pred_oof).round(4))
print('Selisih OOF from scratch - sklearn:',
      round(res_scratch['oof_macro_f1'] - res_sklearn['oof_macro_f1'], 6))

## Diagnostik OOF full-training

In [ ]:
thresholds = np.round(np.arange(0.20, 0.81, 0.01), 2)
fig, axes = plt.subplots(1, 2, figsize=(12, 4.8), layout='constrained')
for name, proba, result, color, style in [
    ('From scratch', oof_scratch, res_scratch, '#126782', '-'),
    ('scikit-learn', oof_sklearn, res_sklearn, '#b45309', '--'),
]:
    axes[0].plot(thresholds, [macro_f1(y, proba >= t) for t in thresholds],
                 label=name, color=color, linestyle=style)
    axes[0].scatter(result['threshold'], result['oof_macro_f1'], color=color, zorder=3)
axes[0].set(title='Macro F1 OOF terhadap threshold', xlabel='Threshold P(kelas 1)',
            ylabel='Macro F1')
axes[0].legend()
axes[1].plot(per_fold['fold'], per_fold['from_scratch'], 'o-',
             color='#126782', label='From scratch')
axes[1].plot(per_fold['fold'], per_fold['sklearn'], 's--',
             color='#b45309', label='scikit-learn')
axes[1].set(title='Macro F1 pada lima fold validasi', xlabel='Fold', ylabel='Macro F1',
            xticks=range(1, N_FOLDS + 1))
axes[1].legend()
for ax in axes:
    ax.grid(axis='y', alpha=0.2)
fig.suptitle(f'SVM {KERNEL}: {len(y):,} baris train bersih, full-training, seed {SEED}\n{best_params}',
             fontsize=12)
FIG_PATH.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(FIG_PATH, dpi=150)
plt.show()

In [ ]:
import json
np.savez_compressed(RUN_DIR / 'oof_complete.npz', y=y, scratch=oof_scratch, sklearn=oof_sklearn)
summary = dict(status='complete', parameters=dict(BASE_PARAMS, **best_params),
               train_rows=len(train), scratch=res_scratch, sklearn=res_sklearn,
               converged_folds=sum(bool(m.converged_) for m in scratch_models),
               warning_count=len(cv_warnings))
(RUN_DIR / 'results.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')
display(comparison.round(4))
print('Run full selesai. Hasil:', RUN_DIR, flush=True)


## Training final dan data test

Satu model from scratch dilatih pada seluruh
train bersih dengan parameter terpilih. Threshold berasal dari hasil OOF.
Bagian ini menambah satu training final setelah evaluasi; bukan mengulang 5-fold.
ID dan format input diperiksa sebelum training dimulai.


In [ ]:
test_raw = pd.read_csv(ROOT / 'data' / 'test.csv', dtype={'id': str})
sample = pd.read_csv(ROOT / 'data' / 'sample_submission.csv', dtype={'id': str})
assert list(sample.columns) == ['id', TARGET]
assert test_raw['id'].notna().all() and test_raw['id'].is_unique
assert sample['id'].is_unique and test_raw['id'].equals(sample['id'])

prep_final = SVMPreprocessor()
X_train = prep_final.fit_transform(train)
X_test = prep_final.transform(test_raw)
assert TARGET not in X_train.columns and 'id' not in X_train.columns
assert X_train.columns.equals(X_test.columns)
assert np.isfinite(X_train.to_numpy()).all() and np.isfinite(X_test.to_numpy()).all()
threshold = res_scratch['threshold']
started = time.perf_counter()
with threadpool_limits(limits=1):
    model_final = SVM(**BASE_PARAMS, **best_params).fit(X_train, y)
print('Perangkat training:', model_final.device_)
print(f'Training final: {time.perf_counter() - started:.1f} detik; '
      f'iterasi: {model_final.n_iter_}; konvergen: {model_final.converged_}; '
      f'threshold: {threshold}')

## Save dan load model

Preprocessor, model, dan threshold disimpan dalam satu bundle lalu dimuat kembali.


In [ ]:
save_model({'preprocessor': prep_final, 'model': model_final,
            'threshold': threshold}, MODEL_PATH)
bundle = load_model(MODEL_PATH)
print('Model tersimpan:', MODEL_PATH.relative_to(ROOT))

## Prediksi test dan ekspor CSV

Data test hanya di-transform dengan preprocessor dari train. Hasil load harus
menghasilkan prediksi yang sama. CSV siap diunggah manual ke kompetisi setelah
semua sel selesai dijalankan; tidak ada submission otomatis.


In [ ]:
with threadpool_limits(limits=1):
    test_pred = bundle['model'].predict(bundle['preprocessor'].transform(test_raw),
                                        threshold=bundle['threshold'])
    expected_pred = model_final.predict(X_test, threshold=threshold)
np.testing.assert_array_equal(test_pred, expected_pred)
assert test_pred.shape == (len(test_raw),) and np.isin(test_pred, [0, 1]).all()
submission = pd.DataFrame({'id': test_raw['id'], TARGET: test_pred})
assert submission.shape == sample.shape and submission.columns.equals(sample.columns)
assert submission['id'].equals(sample['id'])
SUB_PATH.parent.mkdir(parents=True, exist_ok=True)
submission.to_csv(SUB_PATH, index=False)
pd.testing.assert_frame_equal(submission, pd.read_csv(SUB_PATH, dtype={'id': str}))
print('CSV siap:', SUB_PATH.relative_to(ROOT), '| baris:', len(submission))
display(submission.head())